# DIAMOND Voice — خادم الاستنساخ على Colab
1. من القائمة: **Runtime ▸ Change runtime type ▸ T4 GPU** ثم احفظ.
2. شغّل الخلايا بالترتيب (الأولى ثم الثانية).
3. انسخ «رابط الاتصال» الذي يظهر في آخر الخلية الثانية والصقه في الإضافة (قسم ربط بخادم Colab).
4. اترك هذه الخلية تعمل أثناء استخدامك. عند إغلاقها يتوقف الخادم.

⚠ استنسخ صوتك أنت أو صوتًا لديك إذنه فقط.

In [ ]:
import torch
print("GPU:", torch.cuda.is_available() and torch.cuda.get_device_name(0))
!pip -q install chatterbox-tts
!wget -q -O app.zip https://diamond-tv.shop/downloads/voice-studio-local.zip && unzip -qo app.zip
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
print("تم التثبيت")

In [ ]:
import os, sys, re, secrets, subprocess, time
os.environ["DH_TOKEN"] = secrets.token_urlsafe(16)
os.environ["DH_HOST"] = "127.0.0.1"
sys.path.insert(0, "voice-studio-local")
import app
print("الجهاز:", app.DEVICE)
app.start_receiver()
print("تحميل النموذج (أول مرة عدة دقائق)…")
app.get_tts()
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:7861", "--no-autoupdate"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
for line in p.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
    if time.time() - t0 > 90: break
if not url: raise SystemExit("تعذّر إنشاء النفق، أعد تشغيل الخلية")
print("\n===== رابط الاتصال (الصقه في الإضافة) =====")
print(url + "#" + os.environ["DH_TOKEN"])
print("============================================")
print("اترك الخلية تعمل…")
while True: time.sleep(60)